# Crypto Market Beta with CCI30 index

## 1. Config & Data Loading

In [15]:
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Tuple, Any
from scipy import stats as sp_stats
import statsmodels.api as sm
import time
from datetime import datetime, timedelta, timezone

import warnings
warnings.filterwarnings('ignore')

# proprietary files
import helper
import data_handler
import beta_estimators
import portfolio_builder
import analyzer

print('Modules have been loaded!')

Modules have been loaded!


In [2]:
# load daily data
daily_data = data_handler.load_daily_data()

Load D:/Personal/Jobs/prepa interviews/Crypto united/cci30_daily.csv, frequency=daily
Extract feature (market, coin_prices, in_index)


In [3]:
coin_prices_check = data_handler.check_integrity(daily_data['coin_prices'])
print(coin_prices_check)
daily_data['coin_prices'].describe().T

{'nb_rows': 4072, 'dim': (4072, 119), 'min_date': datetime.date(2015, 1, 1), 'max_date': datetime.date(2026, 2, 23), 'nb_duplicates': np.int64(0), 'nb_nan': np.int64(185335)}


,count,mean,std,min,25%,50%,75%,max
0x,3029.0,0.487293,0.383179,0.095275,0.242538,0.331396,0.600621,2.367480
aave,1971.0,170.585886,109.942526,0.516571,82.862453,131.146057,255.497528,632.266479
aeternity,3029.0,0.331611,0.666076,0.003700,0.031505,0.097159,0.211358,4.990970
algorand,2440.0,0.439119,0.443998,0.083864,0.179520,0.250935,0.437264,3.201412
amp,1992.0,0.013562,0.018042,0.001368,0.003496,0.004831,0.011145,0.108954
...,...,...,...,...,...,...,...,...
waves,3029.0,5.065301,6.973207,0.461806,1.214039,2.250676,5.168688,54.612747
wrapped-bitcoin,2582.0,42739.788081,32356.946279,3395.978516,14928.503174,34581.589844,62996.511719,124828.453125
yearn-finance,2045.0,13827.319302,12480.003106,906.710754,5539.221680,7538.245605,20653.437500,82745.187500
zcash,3029.0,106.799300,116.807149,18.293577,37.023022,58.073483,129.828995,880.760986


In [5]:
# study the number of consituents (CCI30 index)
daily_data['in_index'].sum(axis=1).describe()

count    4072.000000
mean       26.523084
std         5.387601
min        11.000000
25%        26.000000
50%        29.000000
75%        30.000000
max        30.000000
dtype: float64

In [6]:
# 1. Check n_constituents over time
n_constituents = daily_data['in_index'].sum(axis=1)
print(f'Constituents over time: {n_constituents.resample('YE').mean().round(1)}')

n_constituents.to_csv(f'n_constituents series - {datetime.now().strftime('%Y-%m-%d %H%M%S')}.csv', sep=';')

Constituents over time: Date
2015-12-31    13.8
2016-12-31    18.3
2017-12-31    23.5
2018-12-31    28.5
2019-12-31    29.5
2020-12-31    29.0
2021-12-31    29.5
2022-12-31    30.0
2023-12-31    30.0
2024-12-31    29.8
2025-12-31    29.7
2026-12-31    28.0
Freq: YE-DEC, dtype: float64


In [7]:
sample_coins_returns = daily_data['coins_returns']

# 2. Empirical study

## 2.1 Compute beta for each constituent across many windows and for several estimators

In [8]:
# define & build beta
estimators=['ols', 'sw', 'wls', 'vr', 'vck']
windows = [365, 180, 90, 30]

In [9]:
daily_estimators_betas = beta_estimators.compute_multi_windows_beta(True, daily_data['in_index'], sample_coins_returns, daily_data['market_returns'], daily_data['market_ohlc'], windows, estimators=estimators) #estimators=['ols', 'sw', 'wls', 'vr', 'vck']
daily_realized_betas = beta_estimators.compute_rolling_realized_betas_all_windows(True, daily_data['in_index'], sample_coins_returns, daily_data['market_returns'], windows)

sample_coins_returns_masked = data_handler.mask_out_prices_no_constituents(sample_coins_returns, daily_data['in_index'])
eom_estimator_betas, eom_realized_betas = analyzer.eom_resampling(sample_coins_returns_masked, daily_data['market_returns'], daily_estimators_betas, daily_realized_betas, frequency=helper.ESTIMATION_FREQ.DAILY)
if len(eom_estimator_betas) < 1: print('error: eom_estimator_betas is empty')
if len(eom_realized_betas) < 1: print('error: eom_realized_betas is empty')

error during ols for coin [aeternity] -> after mask-out according in_index df, res is empty
error during sw for coin [aeternity] -> after mask-out according in_index df, res is empty
error during wls for coin [aeternity] -> after mask-out according in_index df, res is empty
error during vr for coin [aeternity] -> after mask-out according in_index df, res is empty
20/595 3.361344537815126%
error during ols for coin [ardor] -> after mask-out according in_index df, res is empty
error during sw for coin [ardor] -> after mask-out according in_index df, res is empty
error during wls for coin [ardor] -> after mask-out according in_index df, res is empty
error during vr for coin [ardor] -> after mask-out according in_index df, res is empty
error during ols for coin [ark] -> after mask-out according in_index df, res is empty
error during sw for coin [ark] -> after mask-out according in_index df, res is empty
error during wls for coin [ark] -> after mask-out according in_index df, res is empty
e

## 2.2 Compute the pooled panle regression according Sila (2025)

In [10]:
# pooled panel regression
res_pooled_panel_regression = analyzer.compute_pooled_panel_regression(eom_estimator_betas, eom_realized_betas)
if res_pooled_panel_regression.empty: print('error during compute_pooled_panel_regression')
else: print(res_pooled_panel_regression)

res_pooled_panel_regression.to_csv(f'pooled_panel_regression - results - {datetime.now().strftime('%Y-%m-%d %H%M%S')}.csv', sep=';')

w=365 estimator: ols: ethena is present in estimated_betas but not in realized_betas, no added to the pooled panel
w=365 estimator: ols: mantle is present in estimated_betas but not in realized_betas, no added to the pooled panel
w=365 estimator: ols: mantra is present in estimated_betas but not in realized_betas, no added to the pooled panel
w=365 ols: lead pooled panel regression for 2112 rows
w=365 estimator: sw: ethena is present in estimated_betas but not in realized_betas, no added to the pooled panel
w=365 estimator: sw: mantle is present in estimated_betas but not in realized_betas, no added to the pooled panel
w=365 estimator: sw: mantra is present in estimated_betas but not in realized_betas, no added to the pooled panel
w=365 sw: lead pooled panel regression for 2112 rows
w=365 estimator: wls: ethena is present in estimated_betas but not in realized_betas, no added to the pooled panel
w=365 estimator: wls: mantle is present in estimated_betas but not in realized_betas, no ad

## 2.3 Analyze beta dispersion

In [20]:
# compute stats about avg beta for each window / estimator
#estimator_methds = res_pooled_panel_regression.index.get_level_values('estimator').unique()
res_dispersion_beta = analyzer.analyze_beta_dispersion(estimators, daily_estimators_betas, daily_data['market_returns'], rolling_window=60)
if res_dispersion_beta.empty: print('error during analyze_beta_dispersion')
else: print(res_dispersion_beta)
res_dispersion_beta.to_csv(f'analyze_beta_dispersion - results - {datetime.now().strftime('%Y-%m-%d %H%M%S')}.csv', sep=';')

                  vol_beta_high_vol  vol_beta_low_vol  \
window estimator                                        
30     ols                 0.334397          0.586091   
       sw                  0.287810          0.397393   
       vck                 0.266887          0.393774   
       vr                  0.256630          0.315281   
       wls                 0.343208          0.600585   
90     ols                 0.265770          0.445296   
       sw                  0.232168          0.325742   
       vck                 0.229566          0.336700   
       vr                  0.202757          0.278733   
       wls                 0.267163          0.458553   
180    ols                 0.218374          0.371048   
       sw                  0.199343          0.284278   
       vck                 0.201702          0.299412   
       vr                  0.153332          0.258688   
       wls                 0.225777          0.384690   
365    ols                 0.17

## 2.4 Build the transition matrix

Following the methodology from "Beta stability and portfolio formation" (Levy 1994 / Brooks, Faff & Lee 1998):

    1. At each month-end, rank all CCI30 coins by beta -> assign to quintiles (Q1 = lowest, Q5 = highest)
    2. At next month-end, re-rank -> new quintiles
    3. Count transitions: how many coins moved from quintile i to quintile j
    4. Normalize rows to probabilities

    - Strong diagonal (> 0.35): STABLE — past beta predicts future beta
    - Weak diagonal (~0.20): UNSTABLE — betas are random walks
    - Corner stickiness (Q1, Q5 > Q3): extreme betas are more persistent

    Returns:
    - transition matrix (count), transition probabilities, dictionary of stats:
    
    - diagonal_mean: average probability of staying in same quintile
    - diagonal_values: per-quintile stickiness
    - off_diagonal_mean: average probability of moving
    - extreme_stickiness: average of Q1 and Q5 diagonal (corners)
    - middle_stickiness: Q3 diagonal

In [21]:
# compute matrix transition & statistics
trans_matrices, trans_prob_matrices, all_stats, res_transition_df = analyzer.build_transition_matrix_all(daily_estimators_betas, n_quantiles=5, period='M')
if res_transition_df.empty: print('error during build_transition_matrix_all')
else: print(res_transition_df.round(2))

res_transition_df.to_csv(f'transiton matrix - results - {datetime.now().strftime('%Y-%m-%d %H%M%S')}.csv', sep=';')

                  diagonal_mean  off_diagonal_mean  extreme_stickiness  \
window estimator                                                         
30     ols                 0.37               0.16                0.49   
       sw                  0.38               0.16                0.50   
       vck                 0.36               0.16                0.50   
       vr                  0.37               0.16                0.49   
       wls                 0.37               0.16                0.47   
90     ols                 0.58               0.10                0.73   
       sw                  0.59               0.10                0.74   
       vck                 0.58               0.10                0.73   
       vr                  0.58               0.10                0.73   
       wls                 0.57               0.11                0.70   
180    ols                 0.70               0.07                0.82   
       sw                  0.70       

# 3. portfolio
## 3.1 Build the EW portfolio

Build equally-weighted portfolio returns from CCI30 constituents.
    At each date, average the returns of coins that are in the index.
    
    r_EW(t) = (1/N) × Σᵢ r_i(t)

In [22]:
# ************************************ build ew portfolio ************************************************
ew_returns = portfolio_builder.build_ew_portfolio_returns(sample_coins_returns, daily_data['in_index'])
ew_returns_df = ew_returns.to_frame()

daily_ew_estimators_betas = beta_estimators.compute_multi_windows_beta(False, daily_data['in_index'], ew_returns_df, daily_data['market_returns'], daily_data['market_ohlc'], windows, estimators=estimators) #estimators=['ols', 'sw', 'wls', 'vr', 'vck']
daily_ew_realized_betas = beta_estimators.compute_rolling_realized_betas_all_windows(False, daily_data['in_index'], ew_returns_df, daily_data['market_returns'], windows)

eom_ew_estimator_betas, eom_ew_realized_betas = analyzer.eom_resampling(ew_returns_df, daily_data['market_returns'], daily_ew_estimators_betas, daily_ew_realized_betas, frequency=helper.ESTIMATION_FREQ.DAILY)
if len(eom_ew_estimator_betas) < 1: print('error: eom_ew_estimator_betas is empty')
if len(eom_ew_realized_betas) < 1: print('error: eom_ew_realized_betas is empty')

## 3.2 Save estimated and realised beta

In [23]:
# save dataframe
for window in windows:
    for estimator in estimators:
        eom_ew_estimator_betas[window][estimator]['ew_portfolio'].to_csv(f'EW - {window} {estimator} estimated {datetime.now().strftime('%Y-%m-%d %H%M%S')}.csv', sep=";")
    eom_ew_realized_betas[window]['ew_portfolio'].to_csv(f'EW - {window} realized {datetime.now().strftime('%Y-%m-%d %H%M%S')}.csv', sep=";")

## 3.3 Compute the pooled panel regression

In [24]:
res_pooled_panel_regression = analyzer.compute_pooled_panel_regression(eom_ew_estimator_betas, eom_ew_realized_betas)
if res_pooled_panel_regression.empty: print('error during compute_pooled_panel_regression')
else: print(res_pooled_panel_regression)

res_pooled_panel_regression.to_csv(f'EW - pooled_panel_regression - results - {datetime.now().strftime('%Y-%m-%d %H%M%S')}.csv', sep=';')

w=365 ols: lead pooled panel regression for 109 rows
w=365 sw: lead pooled panel regression for 109 rows
w=365 wls: lead pooled panel regression for 109 rows
w=365 vr: lead pooled panel regression for 109 rows
w=365 vck: lead pooled panel regression for 109 rows
w=180 ols: lead pooled panel regression for 122 rows
w=180 sw: lead pooled panel regression for 122 rows
w=180 wls: lead pooled panel regression for 122 rows
w=180 vr: lead pooled panel regression for 122 rows
w=180 vck: lead pooled panel regression for 122 rows
w=90 ols: lead pooled panel regression for 127 rows
w=90 sw: lead pooled panel regression for 127 rows
w=90 wls: lead pooled panel regression for 127 rows
w=90 vr: lead pooled panel regression for 127 rows
w=90 vck: lead pooled panel regression for 127 rows
w=30 ols: lead pooled panel regression for 132 rows
w=30 sw: lead pooled panel regression for 132 rows
w=30 wls: lead pooled panel regression for 132 rows
w=30 vr: lead pooled panel regression for 132 rows
w=30 vck: 

## 3.4 Analyze beta dispersion

In [25]:
# compute stats about avg beta for each window / estimator
#estimator_methds = res_pooled_panel_regression.index.get_level_values('estimator').unique()
res_dispersion_beta = analyzer.analyze_beta_dispersion(estimators, daily_ew_estimators_betas, daily_data['market_returns'], rolling_window=60)
if res_dispersion_beta.empty: print('error during analyze_beta_dispersion')
else: print(res_dispersion_beta)
res_dispersion_beta.to_csv(f'EW - analyze_beta_dispersion - results - {datetime.now().strftime('%Y-%m-%d %H%M%S')}.csv', sep=';')

                  vol_beta_high_vol  vol_beta_low_vol  \
window estimator                                        
30     ols                      NaN               NaN   
       sw                       NaN               NaN   
       vck                      NaN               NaN   
       vr                       NaN               NaN   
       wls                      NaN               NaN   
90     ols                      NaN               NaN   
       sw                       NaN               NaN   
       vck                      NaN               NaN   
       vr                       NaN               NaN   
       wls                      NaN               NaN   
180    ols                      NaN               NaN   
       sw                       NaN               NaN   
       vck                      NaN               NaN   
       vr                       NaN               NaN   
       wls                      NaN               NaN   
365    ols                     